# Genie Cache & Queue — Demo

The Genie API has a hard limit of **5 queries per minute per workspace**.
This notebook fires 7 queries **in parallel** to demonstrate the problem and how the app solves it.

**The code is 100% identical** — only the base URL changes (`GENIE_HOST` → `APP_HOST`).

```
start-conversation  →  poll get-message  →  result
```

## Setup

1. Copy `.env.example` to `/Workspace/Users/<your-email>/.env`
2. Fill in your values (workspace URL, Space ID(s), App URL, Service Token, etc.)
3. Run the cells below

**Prerequisite:** The Databricks Apps proxy requires an **OAuth token** (JWT). In a workspace notebook the SDK obtains it automatically.

**Multi-Space:** Optionally set `SPACE_2` + `SPACE_2_NAME` to demonstrate cache isolation across Genie Spaces.

In [0]:
%pip install python-dotenv -q

In [ ]:
import requests, time, json, os

from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

# --- Load .env from your Workspace home directory ---
# Copy .env.example to /Workspace/Users/<your-email>/.env and fill in your values
try:
    username = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    )
except Exception:
    username = os.getenv("USER", "<your-email>")

ENV_PATH = f"/Workspace/Users/{username}/.env"
load_dotenv(ENV_PATH, override=True)
print(f"Loaded env from {ENV_PATH}")

# --- Token (SDK auto-detects in workspace notebooks) ---
try:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    TOKEN = w.config.token
    print("Token obtained via Databricks SDK")
except Exception:
    TOKEN = os.getenv("DATABRICKS_TOKEN", "")

assert TOKEN, "TOKEN is empty — set DATABRICKS_TOKEN in your .env"

# --- Endpoints & Spaces ---
GENIE_HOST = os.getenv("GENIE_HOST", "")
APP_HOST   = os.getenv("APP_HOST", "").rstrip("/")

# Primary space (required)
SPACE        = os.getenv("SPACE", "")
SPACE_NAME   = os.getenv("SPACE_NAME", "Primary Space")

# Secondary space (optional — for multi-space demo)
SPACE_2      = os.getenv("SPACE_2", "")
SPACE_2_NAME = os.getenv("SPACE_2_NAME", "Secondary Space")

H = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# --- App Config from .env ---
LAKEBASE_SERVICE_TOKEN   = os.getenv("LAKEBASE_SERVICE_TOKEN", "")
SQL_WAREHOUSE_ID         = os.getenv("SQL_WAREHOUSE_ID", "")
LAKEBASE_INSTANCE_NAME   = os.getenv("LAKEBASE_INSTANCE_NAME", "")
LAKEBASE_SCHEMA          = os.getenv("LAKEBASE_SCHEMA", "public")
CACHE_TABLE_NAME         = os.getenv("CACHE_TABLE_NAME", "cached_queries")
SIMILARITY_THRESHOLD     = float(os.getenv("SIMILARITY_THRESHOLD", "0.92"))
CACHE_TTL_SECONDS        = int(os.getenv("CACHE_TTL_SECONDS", "86400"))
MAX_QUERIES_PER_MINUTE   = int(os.getenv("MAX_QUERIES_PER_MINUTE", "5"))

# Build genie_spaces list (multi-space config)
genie_spaces = [{"id": SPACE, "name": SPACE_NAME}]
if SPACE_2:
    genie_spaces.append({"id": SPACE_2, "name": SPACE_2_NAME})

APP_CONFIG = {
    "lakebase_service_token": LAKEBASE_SERVICE_TOKEN,
    "genie_spaces": genie_spaces,
    "genie_space_id": SPACE,  # backward compat — default active space
    "sql_warehouse_id": SQL_WAREHOUSE_ID,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "cache_ttl_seconds": CACHE_TTL_SECONDS,
    "max_queries_per_minute": MAX_QUERIES_PER_MINUTE,
    "storage_backend": "lakebase",
    "lakebase_instance_name": LAKEBASE_INSTANCE_NAME,
    "lakebase_schema": LAKEBASE_SCHEMA,
    "cache_table_name": CACHE_TABLE_NAME,
}

# Health check
r = requests.get(f"{APP_HOST}/api/health", headers=H, timeout=10)
assert r.status_code == 200 and "healthy" in r.text, f"App unreachable (HTTP {r.status_code}: {r.text[:100]})"

# Configure the App (same settings as the UI Settings tab)
r = requests.put(f"{APP_HOST}/api/config", headers=H, json=APP_CONFIG)
assert r.status_code == 200, f"Config failed: {r.status_code} {r.text[:200]}"

# Queries — replace with questions relevant to your primary Genie Space
questions = [
    "What are the top 3 nations by total revenue?",
    "How many orders were placed in January 1994?",
    "What is the average account balance of customers in the BUILDING segment?",
    "Which supplier has the most parts?",
    "Total number of lineitems with quantity greater than 40",
    "Revenue by year for the ASIA region",
    "How many distinct part types exist?",
]

# Queries for secondary space (optional — replace with relevant questions)
questions_2 = [
    "What is the average trip distance?",
    "How many trips were taken in January 2015?",
    "What is the most common payment type?",
]

spaces_label = ", ".join(f"{s['name']} ({s['id'][:8]}...)" for s in genie_spaces)
print(f"OK | Token: ...{TOKEN[-4:]} | Spaces: {spaces_label} | {len(questions)} queries")

In [ ]:
# =============================================================================
# Same function for direct Genie and via App — only base_url changes.
# Full flow: start-conversation → poll get-message → query-result (data)
# =============================================================================

def genie_query(base_url, question, space_id=None):
    """Full flow identical to a real app consuming the Genie API.
    1. POST start-conversation
    2. Poll GET get-message until COMPLETED
    3. GET query-result to get the data"""

    s = space_id or SPACE

    # 1. Start conversation
    r = requests.post(
        f"{base_url}/api/2.0/genie/spaces/{s}/start-conversation",
        headers=H, json={"content": question}, timeout=180,
    )
    if r.status_code == 429:
        return {"status": "429", "sql": None, "data": None, "from_cache": False}
    if r.status_code != 200:
        return {"status": f"HTTP {r.status_code}", "sql": None, "data": None, "from_cache": False}

    data = r.json()
    conv_id = data.get("conversation_id", "")
    msg_id  = data.get("message_id", "")
    from_cache = data.get("status") == "COMPLETED"  # cache hit returns COMPLETED immediately

    # 2. If not COMPLETED, poll get-message
    if data.get("status") != "COMPLETED" or not data.get("attachments"):
        from_cache = False
        for _ in range(90):
            time.sleep(2)
            r2 = requests.get(
                f"{base_url}/api/2.0/genie/spaces/{s}/conversations/{conv_id}/messages/{msg_id}",
                headers=H, timeout=30,
            )
            if r2.status_code != 200:
                continue
            data = r2.json()
            if data.get("status") == "COMPLETED":
                break
            if data.get("status") in ("FAILED", "CANCELLED"):
                return {"status": data["status"], "sql": None, "data": None, "from_cache": False}
        else:
            return {"status": "TIMEOUT", "sql": None, "data": None, "from_cache": False}

    if data.get("status") != "COMPLETED":
        return {"status": data.get("status", "UNKNOWN"), "sql": None, "data": None, "from_cache": False}

    # 3. Extract SQL and attachment_id
    sql = None
    query_att_id = None
    for att in data.get("attachments", []):
        if not isinstance(att, dict):
            continue
        q = att.get("query")
        if q:
            sql = q.get("query") or q.get("sql")
            query_att_id = att.get("attachment_id")
            break

    if not sql:
        return {"status": "COMPLETED", "sql": None, "data": None, "from_cache": from_cache}

    # 4. Get query-result for actual data
    result_data = None
    if query_att_id:
        try:
            r3 = requests.get(
                f"{base_url}/api/2.0/genie/spaces/{s}/conversations/{conv_id}"
                f"/messages/{msg_id}/attachments/{query_att_id}/query-result",
                headers=H, timeout=60,
            )
            if r3.status_code == 200:
                qr = r3.json()
                stmt = qr.get("statement_response", qr)
                result = stmt.get("result") or {}
                rows = result.get("data_array", [])
                row_count = result.get("row_count", len(rows))
                result_data = {"row_count": row_count, "rows": rows[:5]}
        except Exception:
            pass

    return {"status": "COMPLETED", "sql": sql, "data": result_data, "from_cache": from_cache}


def run_parallel(label, base_url, qs=None, space_id=None):
    """Fire all queries in parallel and display full results."""
    query_list = qs or questions
    print(f"\n{'='*90}")
    print(f"  {label}")
    print(f"{'='*90}")
    results = [None] * len(query_list)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(query_list)) as pool:
        futs = {pool.submit(genie_query, base_url, q, space_id): i for i, q in enumerate(query_list)}
        for f in as_completed(futs):
            idx = futs[f]
            results[idx] = f.result()
    total = time.time() - t0

    for i, q in enumerate(query_list):
        r = results[i]
        tag = "CACHE" if r["from_cache"] else ("429" if r["status"]=="429" else "GENIE")
        print(f"\n  [{i+1}/{len(query_list)}] {tag:>5s} | {r['status']}")
        print(f"  Question: {q}")
        if r["sql"]:
            print(f"  SQL:      {r['sql'][:120]}")
        if r["data"]:
            print(f"  Data:     {r['data']['row_count']} rows")
            for row in r["data"]["rows"][:3]:
                print(f"            {row}")
        elif r["status"] not in ("429",):
            print(f"  (no data)")

    print(f"\n  Total time: {total:.1f}s")
    return results, total

print("Functions defined. Ready for scenarios.")

## Scenario 1: Direct to Genie (7 in parallel)

Same code, `base_url = GENIE_HOST`. 7 simultaneous queries → 5/min limit → **429 Too Many Requests**.

In [0]:
direct, d_time = run_parallel("SCENARIO 1 — Direct to Genie (7 in parallel)", GENIE_HOST)

ok = sum(1 for r in direct if r["sql"])
blocked = sum(1 for r in direct if r["status"] == "429")
print(f"\nResult: {ok} completed, {blocked} blocked (429)")

## Scenario 2a: Via App — first round (7 in parallel)

**Same code**, only swap `base_url = APP_HOST`. The App manages rate limiting, queue, and retry internally.

In [0]:
app1, a1_time = run_parallel("SCENARIO 2a — Via App, first round (7 in parallel)", APP_HOST)

genie_ok = sum(1 for r in app1 if not r.get("from_cache") and r["sql"])
cache_ok = sum(1 for r in app1 if r.get("from_cache"))
failed   = sum(1 for r in app1 if r["status"] not in ("COMPLETED",))
blocked  = sum(1 for r in app1 if r["status"] == "429")
print(f"\nResult: {genie_ok} via Genie, {cache_ok} from cache, {failed} failures, {blocked} 429 errors")

## Scenario 2b: Via App — second round (all from cache)

Same queries again. All should come from the **semantic cache** (instant response).

In [0]:
app2, a2_time = run_parallel("SCENARIO 2b — Via App, second round (cache)", APP_HOST)

cache_count = sum(1 for r in app2 if r.get("from_cache"))
print(f"\nResult: {cache_count}/{len(questions)} from cache | {a2_time:.1f}s vs {d_time:.1f}s direct")

## Scenario 3: Multi-Space Cache Isolation (optional)

If `SPACE_2` is configured, queries to a **different space** should be **cache misses** — even if similar questions were already cached for the primary space. This proves the cache is partitioned by space.

In [ ]:
if not SPACE_2:
    print("SPACE_2 not configured — skipping multi-space scenarios.")
    print("Set SPACE_2 and SPACE_2_NAME in your .env to enable this test.")
    multi_1 = multi_2 = None
    m1_time = m2_time = 0
else:
    # First round: all cache misses (different space)
    multi_1, m1_time = run_parallel(
        f"SCENARIO 3a — Space 2 ({SPACE_2_NAME}), first round",
        APP_HOST, qs=questions_2, space_id=SPACE_2
    )
    genie_ok = sum(1 for r in multi_1 if not r.get("from_cache") and r["sql"])
    cache_ok = sum(1 for r in multi_1 if r.get("from_cache"))
    print(f"\nResult: {genie_ok} via Genie, {cache_ok} from cache (expected: all Genie, 0 cache)")

    # Second round: all from cache
    multi_2, m2_time = run_parallel(
        f"SCENARIO 3b — Space 2 ({SPACE_2_NAME}), second round (cache)",
        APP_HOST, qs=questions_2, space_id=SPACE_2
    )
    cache_count = sum(1 for r in multi_2 if r.get("from_cache"))
    print(f"\nResult: {cache_count}/{len(questions_2)} from cache | {m2_time:.1f}s")

In [ ]:
# Comparison table
def _tag(r):
    if r["from_cache"]: return "CACHE"
    if r["status"] == "429": return "429"
    if r["sql"]: return "OK"
    return r["status"][:6]

print(f"\n{'='*70}")
print(f"  FINAL COMPARISON — {SPACE_NAME}")
print(f"{'='*70}")
print(f"\n{'Question':32s} | {'Direct':>8s} | {'App 1st':>8s} | {'App 2nd':>8s}")
print("-" * 68)
for i in range(len(questions)):
    print(f"  {questions[i][:30]:30s} | {_tag(direct[i]):>8s} | {_tag(app1[i]):>8s} | {_tag(app2[i]):>8s}")
print(f"  {'TIME':30s} | {d_time:>7.1f}s | {a1_time:>7.1f}s | {a2_time:>7.1f}s")

# Multi-space summary (if available)
if SPACE_2 and multi_1 and multi_2:
    print(f"\n{'='*70}")
    print(f"  MULTI-SPACE — {SPACE_2_NAME}")
    print(f"{'='*70}")
    print(f"\n{'Question':32s} | {'1st run':>8s} | {'2nd run':>8s}")
    print("-" * 55)
    for i in range(len(questions_2)):
        print(f"  {questions_2[i][:30]:30s} | {_tag(multi_1[i]):>8s} | {_tag(multi_2[i]):>8s}")
    print(f"  {'TIME':30s} | {m1_time:>7.1f}s | {m2_time:>7.1f}s")

print(f"\n{'='*70}")
print(f"  SUMMARY")
print(f"{'='*70}")
print(f"  Direct:  {sum(1 for r in direct if r['status']=='429')} queries blocked (429)")
print(f"  App 1st: {sum(1 for r in app1 if r['sql'])}/{len(questions)} completed (zero 429)")
print(f"  App 2nd: {sum(1 for r in app2 if r['from_cache'])}/{len(questions)} from cache ({a2_time:.1f}s)")
if SPACE_2 and multi_1 and multi_2:
    print(f"  Space 2: {sum(1 for r in multi_1 if r['sql'])}/{len(questions_2)} completed, "
          f"{sum(1 for r in multi_2 if r['from_cache'])}/{len(questions_2)} cached ({m2_time:.1f}s)")

# Cache entries via unified API
try:
    entries = requests.post(f"{APP_HOST}/api/cache", headers=H,
        json={"config": {"storage_backend": "lakebase",
              "lakebase_instance_name": LAKEBASE_INSTANCE_NAME,
              "lakebase_schema": LAKEBASE_SCHEMA, "cache_table_name": CACHE_TABLE_NAME,
              "auth_mode": "user", "user_pat": TOKEN}},
        timeout=10).json()
    if entries:
        print(f"\n--- Cache ({len(entries)} entries) ---")
        for e in entries[:10]:
            sid = e.get("genie_space_id", "")[:8]
            print(f"  [{e.get('use_count',0)}x] ({sid}...) {e['query_text'][:40]:40s} | {e.get('sql_query','')[:45]}")
except Exception as ex:
    print(f"  (cache unavailable: {ex})")